# IMPORT

In [1]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

import os
import gc
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [2]:
STATION_CLUSTER_FOLDER = Path("../data/station_clusters")
DATASET_FOLDER = Path("../data/clean_data_csv")
MODEL_OUTPUT_FOLDER = Path("../data/cluster_models")
PLOT_FOLDER = Path("../data/model_plots")

folders = [STATION_CLUSTER_FOLDER, DATASET_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [ ]:
FEATURE_NAME = "11_features" # "11_features" or "5_features"
HAS_BIKE_LAG = True
LAGGED_NAME = "lagged" if HAS_BIKE_LAG else "not_lagged"
CLUSTER_FILE_NAME = f"station_clusters_2_42_{FEATURE_NAME}"

# LOAD DATA AND CLUSTERS

In [4]:
df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/{CLUSTER_FILE_NAME}.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

Rows: (6800816, 21)
Clusters: cluster
0    3457266
1    3343550
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'total_bikes_lag1',
       'total_bikes_lag2', 'split', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [5]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",

    # Station context
    "capacity"
]
if HAS_BIKE_LAG:
    FEATURES = [
        # Time
        "sin_hour",
        "cos_hour",
        "is_weekend",

        # Weather
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code",
        "is_day",

        # Station context
        "capacity",

        # Bike lag
        "total_bikes_lag1", 
        "total_bikes_lag2"
    ]

TARGET = "total_bikes_now"


# TRAIN MODELS PER CLUSTER

In [34]:
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n - Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    if HAS_BIKE_LAG:
        if "total_bikes_lag1" not in df_c.columns:
            df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
            df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
            df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Select train/val using the split column
    train = df_c[df_c["split"] == 0]
    val   = df_c[df_c["split"] == 1]

    X_train, y_train = train[FEATURES], train[TARGET]
    X_val, y_val     = val[FEATURES], val[TARGET]

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    # Train and evaluate on validation set
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        r2 = r2_score(y_val, preds)
        mae = mean_absolute_error(y_val, preds)

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} -> R^2 = {r2:.3f} | MAE = {mae:.2f}")



 - Training cluster 0
  LinearRegression → R^2=0.938 | MAE=1.02
  SGDRegressor     → R^2=0.938 | MAE=1.01
  RandomForest     → R^2=0.943 | MAE=0.93

 - Training cluster 1
  LinearRegression → R^2=0.968 | MAE=1.25
  SGDRegressor     → R^2=0.968 | MAE=1.27
  RandomForest     → R^2=0.970 | MAE=1.16


# COMPARE RESULTS

In [35]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,n_samples
2,0,RandomForest,0.942831,0.931938,3457266
0,0,LinearRegression,0.938136,1.018081,3457266
1,0,SGDRegressor,0.938062,1.011218,3457266
5,1,RandomForest,0.970275,1.159158,3343550
3,1,LinearRegression,0.968200,1.252432,3343550
4,1,SGDRegressor,0.968106,1.270941,3343550


# GET BEST MODEL PER CLUSTER

In [36]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE BEST MODEL PER CLUSTER

In [ ]:
for cluster_id in best_models:
    df_c = df[df["cluster"] == cluster_id]
    X = df_c[FEATURES]
    y = df_c[TARGET]

    model_name = best_models[cluster_id]

    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ])
    else:
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        )

    model.fit(X, y)
    joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/{model_name}_model_cluster_{cluster_id}_{FEATURE_NAME}_{LAGGED_NAME}.pkl")

print("Models saved per cluster")


Models saved per cluster


# QUICK TEST ON THE MODELS

In [10]:
# LOAD THE MODELS
pred_models = {}
for i in range(2):
    pred_models[i] = joblib.load(f"{MODEL_OUTPUT_FOLDER}/model_cluster_{i}_{FEATURE_NAME}_{LAGGED_NAME}.pkl")

In [11]:
df_val = df[df["split"] == 1].copy()
del df
gc.collect()

NameError: name 'df' is not defined

In [ ]:
def plot_station(args):
    cluster_id, station_code, df_station = args

    df_station = df_station.sort_values("timestamp")

    X = df_station[FEATURES]
    y = df_station[TARGET]

    model = pred_models[cluster_id]
    preds = model.predict(X)

    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)

    """
    plt.figure(figsize=(14, 5))
    plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
    plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

    plt.title(
        f"Cluster {cluster_id} | Station {station_code}\n"
        f"Model: {type(model).__name__}"
    )
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        f"{PLOT_FOLDER}/cluster_{cluster_id}_station_{station_code}_{FEATURE_NAME}_{LAGGED_NAME}.png"
    )
    plt.close()
    """

    return {
        "cluster": cluster_id,
        "station": station_code,
        "model": type(model).__name__,
        "r2": r2,
        "mae": mae,
        "n_samples": len(df_station),
    }


In [8]:
os.cpu_count()

16

In [12]:
jobs = [
    (cluster_id, station_code, df_station)
    for (cluster_id, station_code), df_station
    in df_val.groupby(["cluster", "station_code"], sort=False)
]

results = []

with ThreadPoolExecutor(max_workers=8) as ex:
    for res in tqdm(ex.map(plot_station, jobs), total=len(jobs)):
        results.append(res)

results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{PLOT_FOLDER}/all_stations_results_{FEATURE_NAME}_{LAGGED_NAME}.csv",
    index=False
)

100%|██████████| 1436/1436 [3:31:05<00:00,  8.82s/it]    
